# Chapter 04 제출 답안 양식. pandas로 데이터에 질문하기

> 주 제출물은 실행 완료 Notebook `chapter04/chapter04.ipynb`입니다. 이 양식의 항목을 Notebook의 Markdown 셀로 추가해 작성합니다.

## 0. 제출 정보
- 이름: 이상재
- GitHub ID: sangjae-lee97
- 작성일: 2026-09-09
- 최종 제출 URL: https://github.com/sangjae-lee97/kant-axagent-study/blob/main/llm-data-analysis-course/chapter04/chapter04.ipynb

## 1. 질문과 필요한 데이터 선택

### 내가 확인하려는 질문

completed 주문을 기준으로 카테고리별, 상품별, 월별, 고객별 판매 현황을 요약하고,
각 집계 결과의 total_sales가 동일한 전체 판매 금액을 나타내는지 확인한다.

### 사용한 파일/컬럼

- customers.csv
  - customer_id
  - city

- products.csv
  - product_id
  - product_name
  - category

- orders.csv
  - order_id
  - customer_id
  - order_date
  - order_status

- order_items.csv
  - order_id
  - product_id
  - quantity
  - unit_price


### 결과 관찰

4개의 CSV가 모두 정상적으로 로드되었고,
각 데이터에서 분석에 필요한 컬럼이 존재하는 것을 확인했다.

특히 판매 금액은 order_items의 quantity와 unit_price를 사용해 계산하고,
orders의 order_status를 이용해 completed 주문만 선택해야 했다.

상품별·카테고리별 분석에는 products가 필요했고,
고객별 분석에는 orders의 customer_id와 customers의 고객 속성이 필요했다.

### 나의 해석과 판단

하나의 파일만으로는 모든 질문에 답할 수 없었다.

order_items에는 주문 상세 금액 정보가 있지만 주문 상태와 날짜가 없고,
orders에는 주문 상태와 고객 정보가 있지만 상품명과 카테고리가 없다.

따라서 분석 질문에 따라 여러 데이터를 key를 기준으로 병합할 필요가 있다고 판단했다.

### 업무·분석적 의미

분석을 시작하기 전에 질문에 필요한 데이터와 컬럼을 먼저 정의하면
불필요한 데이터를 사용하지 않고 분석 기준을 명확히 할 수 있다.

또한 어떤 데이터가 어떤 역할을 하는지 구분하면
이후 merge 과정에서 어떤 key를 사용해야 하는지도 판단하기 쉬워진다.

### 한계와 추가 확인 사항

현재 데이터는 샘플 데이터이므로 실제 서비스의 모든 판매 상황을 반영한다고 볼 수 없다.

또한 total_sales는 completed 주문의 quantity × unit_price를 합산한 값이며,
할인, 환불, 배송비, 세금 등의 정보는 현재 데이터에 포함되어 있는지 별도로 확인할 필요가 있다.

## 2. 필터링·정렬·파생 컬럼

- 적용한 필터 조건:
  - customer age >= 30
  - city가 서울 또는 부산
  - 최종 판매 분석에서는 order_status == "completed"

- 정렬 기준:
  - 상품 가격(price) 내림차순
  - 이후 집계 결과에서는 total_sales 기준 내림차순
  - 월별 결과는 order_month 기준 오름차순

- 만든 파생 컬럼:
  - line_total
  - order_month

- `line_total` 계산식:
  - quantity × unit_price

![필터와 파생 컬럼](images/step02_transform.png)

### 결과 관찰

필터 조건을 적용하면 조건을 만족하는 행만 남는 것을 확인했다.

order_items에 quantity × unit_price를 계산하여 line_total 컬럼을 만들었고,
주문 날짜를 datetime으로 변환한 뒤 order_month 컬럼을 생성했다.

최종 판매 집계에서는 completed 상태의 주문만 남겨 분석 범위를 제한했다.

### 나의 해석과 판단

필터 기준이 달라지면 분석 대상 자체가 달라지기 때문에 결과도 달라진다.

예를 들어 completed 주문뿐 아니라 cancelled 주문까지 포함하면
실제로 완료되지 않은 주문의 금액도 total_sales에 포함될 수 있다.

또한 특정 도시나 연령만 선택할 경우 전체 고객을 대상으로 한 결과가 아니라
해당 조건의 고객에 대한 결과로 해석해야 한다.

### 업무·분석적 의미

필터는 단순히 데이터를 줄이는 작업이 아니라
"어떤 데이터를 분석 대상으로 인정할 것인가"를 결정하는 과정이라고 생각했다.

특히 주문 상태처럼 매출 계산에 직접 영향을 주는 조건은
분석 결과의 의미를 결정하므로 명확하게 기록해야 한다.

### 한계와 추가 확인 사항

completed라는 상태가 실제 업무에서 반드시 매출 확정 상태를 의미하는지는
업무 정책을 추가로 확인해야 한다.

또한 line_total은 단순히 quantity × unit_price로 계산했기 때문에
할인, 쿠폰, 환불 등의 요소가 별도로 존재한다면 실제 금액과 차이가 있을 수 있다.


## 3. merge 검증

- 병합한 데이터:
  1. order_items + orders
  2. completed_order_sales + products

- 사용한 key:
  1. order_id
  2. product_id

- `validate` 결과:
  - 두 병합 모두 many_to_one 조건에서 오류가 발생하지 않았다.

- `indicator` 결과:
  - [직접 입력: 예) both만 존재, left_only 0건]

- 병합 전/후 행 수:
  - order_items + orders
    - 병합 전: [직접 입력]
    - 병합 후: [직접 입력]

  - completed_order_sales + products
    - 병합 전: [직접 입력]
    - 병합 후: [직접 입력]

![merge 검증](images/step03_merge.png)

### 결과 관찰

병합 전후의 행 수를 비교한 결과 [같았다 / 차이가 있었다].

indicator 결과에서 [both만 존재했다 / left_only가 몇 건 존재했다].

또한 orders.order_id와 products.product_id의 중복 여부를 확인해
오른쪽 테이블의 병합 key가 고유하게 관리되고 있는지 확인했다.

### 나의 해석과 판단

이번 병합이 안전하다고 판단한 근거는
many_to_one 검증이 통과했고,
병합 전후 행 수가 동일했으며,
indicator에서 미매칭 행이 없었기 때문이다.

즉 하나의 주문 상세 행이 병합 과정에서 여러 행으로 불필요하게 증가하지 않았고,
연결되지 않은 주문 또는 상품도 없다고 판단할 수 있었다.

### 업무·분석적 의미

잘못된 merge가 발생하면 동일한 주문이 여러 번 복제되어
total_sales가 실제보다 크게 계산될 수 있다.

반대로 key가 매칭되지 않아 행이 누락되면
판매 금액이 실제보다 작게 계산될 수 있다.

따라서 merge가 실행되었다는 사실만 확인하는 것이 아니라
관계의 방향, 행 수 변화, 미매칭 여부를 함께 검증해야 한다.

### 한계와 추가 확인 사항

현재 검증에서는 key 중복과 미매칭 여부를 중심으로 확인했다.

실제 업무 데이터에서는 key 값이 형식상 동일하더라도
잘못된 상품이나 고객과 연결되어 있을 가능성까지 검증하려면
추가적인 데이터 품질 검사가 필요할 수 있다.

## 4. completed 주문 범위와 집계

- 분석 범위 정의:
  - order_status가 completed인 주문만 사용

- 카테고리별 결과:
  - category별 total_quantity, total_sales 집계

- 상품별 결과:
  - product_id, product_name, category별 total_quantity, total_sales 집계

- 월별 결과:
  - order_month별 total_sales, 고유 order_count 집계

- 고객별 결과:
  - customer_id별 고유 order_count, total_sales 집계

![핵심 집계 결과](images/step04_groupby.png)

### 결과 관찰

카테고리별로 가장 높은 total_sales를 기록한 카테고리는
[카테고리명]이며 total_sales는 [금액]이었다.

상품별로 가장 높은 판매 금액을 기록한 상품은
[상품명]이며 total_sales는 [금액]이었다.

월별 결과에서는 [월]의 total_sales가 가장 높았고
[월]이 가장 낮았다.

고객별로는 [customer_id] 고객의 total_sales가 가장 높았다.

### 나의 해석과 판단

가장 중요하다고 판단한 결과는 [직접 선택]이다.

그 이유는 단순한 전체 판매 금액보다
어떤 카테고리/상품/고객/기간이 판매 금액에 가장 크게 기여했는지를 확인해야
추가 분석 방향을 정할 수 있기 때문이다.

예를 들어 특정 상품 또는 카테고리에 판매 금액이 집중되어 있다면
해당 상품의 판매 원인을 분석하거나
다른 상품과의 차이를 비교할 수 있다.

### 업무·분석적 의미

이번 결과는 다음 단계에서
상위 판매 상품의 특징 분석,
월별 판매 변화 원인 분석,
고객별 구매 규모 비교 등으로 확장할 수 있다.

이를 통해 프로모션 대상 선정,
상품 구성 개선,
고객 세분화 등의 의사결정에 활용할 수 있다.

다만 현재 단계에서는 원인을 분석한 것이 아니라
판매 현황을 요약한 단계이므로
집계 결과만 보고 인과관계를 단정해서는 안 된다.

### 한계와 추가 확인 사항

이번 total_sales는 completed 주문에 포함된
line_total의 합계이다.

line_total 자체가 quantity × unit_price로 계산되었기 때문에
할인, 쿠폰, 배송비, 세금, 환불, 취소 후 정산 등의 정보가 반영되었는지 알 수 없다.

따라서 현재 total_sales를 회계상의 순매출과 같다고 단정할 수 없다.

## 5. 총합 일치 검증

- 원본 completed `line_total` 합계: 148990000
- 카테고리 합계: 148990000
- 월별 합계: 148990000
- 고객별 합계: 148990000
- 차이 여부: 없음

![총합 검증](images/step05_total_check.png)

### 나의 해석과 판단

completed 주문 상세 데이터의 `line_total` 합계와
카테고리별, 월별, 고객별 집계의 `total_sales` 합계가 모두 148990000으로 동일하게 나왔다.

즉, 서로 다른 기준으로 `groupby()`를 수행했지만
모두 같은 completed 주문 데이터를 기준으로 집계되었으며,
집계 과정에서 금액이 중복 계산되거나 누락되지 않았다고 판단할 수 있다.

총합 검증이 필요한 이유는
코드가 실행되었다는 사실만으로 집계 결과의 정확성을 보장할 수 없기 때문이다.

예를 들어 merge 과정에서 중복 행이 생기면 합계가 실제보다 커질 수 있고,
필터 조건이 잘못되거나 일부 행이 누락되면 합계가 실제보다 작아질 수 있다.

따라서 원본 completed `line_total` 합계와
각 summary 결과의 `total_sales` 합계를 비교하는 것은
분석 결과가 같은 범위를 일관되게 반영하고 있는지 확인하는 중요한 검증 과정이라고 생각했다.

만약 합계가 일치하지 않았다면 다음 순서로 확인했을 것이다.

1. `completed` 필터가 정확히 적용되었는지 확인
2. merge 전후 행 수가 같은지 확인
3. `indicator` 결과에서 미매칭(`left_only`)이 있는지 확인
4. merge key의 중복 여부 확인
5. 월별 집계에서 `order_month` 결측치가 있는지 확인
6. `groupby()` 대상 컬럼과 집계 범위가 같은지 확인

## 6. LLM pandas 코드 검증

- LLM Prompt 요약:
  - completed 주문 기준으로 상품 정보를 병합하고 판매 분석을 진행하는 pandas 코드를 요청함.

- 제안 코드 요약:
  - completed_order_sales와 products를 product_id 기준으로 merge
  - validate="many_to_one"
  - indicator=True 사용

- 실제 컬럼/범위와 맞지 않은 부분:
  - 이전 merge에서 생성된 `_merge` 컬럼이 completed_order_sales에 이미 존재했음.
  - 이 상태에서 다시 indicator=True를 사용하자
    `Cannot use name of an existing column for indicator column`
    오류가 발생함.

- 수정한 내용:
  - 기존 `_merge` 컬럼을 제거하거나,
  - indicator의 이름을 별도로 지정하도록 수정함.

- 최종 판단:
  - 수정 후 사용

  ![LLM 코드 검증](images/step06_llm_validation.png)

### 나의 해석과 판단

LLM이 제안한 코드가 문법적으로 맞더라도
현재 DataFrame의 상태를 완전히 반영하지 못할 수 있다는 것을 확인했다.

이번 경우에도 merge 코드 자체는 일반적으로 올바른 형태였지만,
이전에 생성된 `_merge` 컬럼이 이미 존재하고 있다는 실행 상태까지는 고려하지 못했다.

따라서 LLM 코드가 실행되는지만 보는 것이 아니라
현재 DataFrame의 컬럼,
이전 처리 과정,
merge key 관계,
필터 범위 등을 직접 확인해야 한다고 판단했다.

## 7. Chapter 04 최종 인사이트

### 가장 의미 있다고 생각한 결과 2가지

1. 스포츠 카테고리가 전체 카테고리 중 가장 높은 판매 금액을 기록했다.

스포츠 카테고리의 `total_sales`는 31,743,000원으로 가장 높았고,
`total_quantity`도 295개로 가장 많았다.

상품별 TOP 5에서도 `스포츠 상품 041`이 5,705,000원으로 1위를 기록했고,
`스포츠 상품 009`도 3,860,000원으로 TOP 5에 포함되었다.

따라서 현재 데이터에서는 스포츠 카테고리가
판매 금액과 판매 수량 모두에서 상대적으로 높은 성과를 보였다고 판단했다.

2. 월별 판매 금액은 일정하지 않고 월마다 큰 차이를 보였다.

가장 높은 월은 2025-10으로 `total_sales`가 25,766,000원이었고,
주문 수도 26건으로 가장 많았다.

반대로 2026-07은 `total_sales`가 2,188,000원,
주문 수는 2건으로 가장 낮았다.

월별 결과를 보면 판매 금액이 높은 달은 주문 수도 대체로 많은 편이므로,
월별 판매 차이에는 주문 건수 변화가 영향을 주었을 가능성이 있다고 생각했다.

### 그 결과를 뒷받침하는 수치/표

#### 카테고리별 주요 결과

| category | total_quantity | total_sales |
| --- | ---: | ---: |
| 스포츠 | 295 | 31,743,000 |
| 전자기기 | 259 | 26,400,000 |
| 생활용품 | 272 | 23,915,000 |
| 뷰티 | 223 | 23,383,000 |
| 식품 | 133 | 16,573,000 |
| 도서 | 149 | 16,389,000 |
| 패션 | 111 | 10,587,000 |

#### 상품별 TOP 5

| product_name | category | total_quantity | total_sales |
| --- | --- | ---: | ---: |
| 스포츠 상품 041 | 스포츠 | 35 | 5,705,000 |
| 식품 상품 012 | 식품 | 25 | 4,375,000 |
| 스포츠 상품 009 | 스포츠 | 20 | 3,860,000 |
| 뷰티 상품 072 | 뷰티 | 20 | 3,780,000 |
| 전자기기 상품 071 | 전자기기 | 23 | 3,703,000 |

#### 월별 주요 결과

| order_month | total_sales | order_count |
| --- | ---: | ---: |
| 2025-10 | 25,766,000 | 26 |
| 2026-04 | 17,553,000 | 23 |
| 2026-01 | 17,423,000 | 22 |
| 2025-08 | 15,621,000 | 18 |
| 2026-07 | 2,188,000 | 2 |

#### 고객별 상위 결과

| customer_id | order_count | total_sales | city |
| --- | ---: | ---: | --- |
| 117 | 5 | 4,100,000 | 성남 |
| 102 | 4 | 3,996,000 | 고양 |
| 83 | 4 | 3,880,000 | 수원 |
| 30 | 5 | 3,590,000 | 서울 |
| 40 | 4 | 3,523,000 | 서울 |

### 추가로 확인하고 싶은 질문

스포츠 카테고리의 판매 금액이 높은 이유가
판매 수량이 많기 때문인지, 상품 단가가 높기 때문인지 추가로 확인하고 싶다.

또한 2025-10의 판매 금액과 주문 수가 가장 높았던 이유가
특정 인기 상품의 판매 증가 때문인지,
전체적으로 주문이 증가했기 때문인지도 확인하고 싶다.

고객별 분석에서는 상위 고객의 구매가 특정 상품이나 카테고리에 집중되어 있는지도 추가로 확인해 보고 싶다.

### 현재 결과의 한계

현재 분석은 샘플 데이터의 `completed` 주문만 대상으로 한 집계 결과이기 때문에
실제 시장이나 고객의 일반적인 구매 패턴이라고 단정할 수 없다.

또한 월별 판매 차이가 계절성 때문인지, 프로모션이나 특정 상품의 영향 때문인지는
현재 데이터만으로 알 수 없다.

고객별 결과도 단순히 구매 금액과 주문 수만 집계했기 때문에
고객의 구매 성향이나 재구매 패턴까지 설명하기에는 부족하다.

마지막으로 `total_sales`는 `quantity × unit_price`를 기준으로 계산한 값이므로
할인, 쿠폰, 환불, 배송비, 세금 등이 반영된 회계상 순매출이라고 단정할 수 없다.

# 1. 실행 환경 확인

In [98]:
from course_utils.paths import get_project_root, get_data_dir
import pandas as pd

print("프로젝트 루트 :", get_project_root())
print("데이터 폴더 :", get_data_dir())

프로젝트 루트 : C:\dev\kant-axagent-study\llm-data-analysis-course
데이터 폴더 : C:\dev\kant-axagent-study\llm-data-analysis-course\data


# 2. 4개 CSV 확인하고 불러오기

In [99]:
customers = pd.read_csv(get_data_dir() / "raw/customers.csv")
products = pd.read_csv(get_data_dir() / "raw/products.csv")
orders = pd.read_csv(get_data_dir() / "raw/orders.csv")
order_items = pd.read_csv(get_data_dir() / "raw/order_items.csv")

print(type(customers))
print(type(products))
print(type(orders))
print(type(order_items))

<class 'pandas.DataFrame'>
<class 'pandas.DataFrame'>
<class 'pandas.DataFrame'>
<class 'pandas.DataFrame'>


# 3. 실제 컬럼명 확인하기

In [100]:
datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

for name, df in datasets.items():
    print(name, df.shape, df.columns.tolist())

customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


# 4. 컬럼 선택과 행 필터링

In [101]:
# 필요한 컬럼만 선택
customer_basic = customers[[
    "customer_id",
    "gender",
    "age",
    "city",
]]

In [102]:
# 30세 이상 고객을 선택
customers_over_30 = customers[customers["age"] >= 30]
print(customers_over_30)

     customer_id name gender  age city signup_date
1              2  김정호      F   32   대구  2025-11-02
2              3  이경수      F   61   성남  2024-06-12
3              4  조영호      F   55   울산  2026-04-13
5              6  김지원      F   32   성남  2026-06-27
6              7  이상현      F   53   인천  2024-12-12
..           ...  ...    ...  ...  ...         ...
141          142  하서연      F   32   서울  2023-08-09
142          143  김예은      F   43   대전  2024-01-02
144          145  조정호      M   59   성남  2024-01-02
145          146  김숙자      M   61   성남  2025-12-23
149          150  조미영      M   40   대전  2025-12-04

[111 rows x 6 columns]


In [103]:
# 서울 또는 부산 고객을 선택
city_customers = customers[customers["city"].isin(["서울", "부산"])]
print(city_customers[["name", "city"]])

    name city
4    이예원   부산
8    송지민   서울
14   장정식   서울
15   강보람   부산
29   이민재   서울
39   박예준   서울
41   장성호   부산
44   김명자   부산
47   김예은   서울
53   김지원   부산
65   김재호   서울
67   이정남   서울
68   남진우   서울
73   장윤서   부산
76   김은주   서울
78   이서준   부산
91   김지은   서울
96   주민준   부산
100  박정희   부산
105  배은정   부산
109  이정숙   부산
114  이민재   부산
115  권종수   부산
120  손영자   서울
122  김재현   서울
125  김정수   서울
131  배예지   부산
138  하승현   서울
141  하서연   서울
146  이정남   부산
148  김정자   부산


In [104]:
# 필터링 전에 실제 값 확인하기
print(customers["city"].value_counts())
print(orders["order_status"].value_counts())

city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64
order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64


# 5. 정렬과 파생 컬럼 만들기

In [105]:
products.sort_values("price", ascending = False).head(10)

,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


In [106]:
order_items = order_items.copy()
order_items["line_total"] = (order_items["quantity"] * order_items["unit_price"])
print(order_items.columns)

Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price',
       'line_total'],
      dtype='str')


# 6. orders와 병합하고 검증하기

In [107]:
print("orders.orders_id 중복 수:", orders["order_id"].duplicated().sum())

orders.orders_id 중복 수: 0


In [108]:
order_sales = order_items.merge(
    orders[[
        "order_id",
        "customer_id",
        "order_date",
        "order_status"
    ]],
    on = "order_id",
    how = "left",
    validate = "many_to_one",
    indicator = True
)

In [109]:
print("병합 후 column : ", order_sales.columns)
print("병합 전 행 수:", len(order_items))
print("병합 후 행 수:", len(order_sales))
print(order_sales["_merge"].value_counts())

병합 후 column :  Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price',
       'line_total', 'customer_id', 'order_date', 'order_status', '_merge'],
      dtype='str')
병합 전 행 수: 764
병합 후 행 수: 764
_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64


# 7. 날짜 변환 후 completed 주문만 선택하기

In [110]:
order_sales["order_date"] = pd.to_datetime(
    order_sales["order_date"],
    errors = "coerce"
)

print("날짜 변환 실패: ", order_sales["order_date"].isna().sum())

날짜 변환 실패:  0


In [111]:
completed_order_sales = order_sales[
    order_sales["order_status"] == "completed"
].copy()

In [112]:
completed_order_sales["order_month"] = (
    completed_order_sales["order_date"].dt.to_period("M").astype(str))

In [113]:
completed_order_sales["order_status"].unique()

<StringArray>
['completed']
Length: 1, dtype: str

In [114]:
orders["order_status"].value_counts(dropna=False)

order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

# 8. 병합하고 다시 검증하기

In [115]:
completed_sales_items = completed_order_sales.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator="product_merge_result",
)

In [116]:
print("병합 전:", len(completed_order_sales))
print("병합 후:", len(completed_sales_items))
print(completed_sales_items["_merge"].value_counts())

병합 전: 474
병합 후: 474
_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64


# 9. 카테고리별 상품별 요약표 만들기

In [117]:
category_sales = (
    completed_sales_items
    .groupby("category", as_index = False)
    .agg(
        total_quantity = ("quantity", "sum"),
        total_sales = ("line_total", "sum"),
    )
    .sort_values("total_sales", ascending = False)
)
print(category_sales)

product_sales = (
    completed_sales_items
    .groupby(
        ["product_id", "product_name", "category"],
        as_index=False,
    )
    .agg(
        total_quantity=("quantity", "sum"),
        total_sales=("line_total", "sum"),
    )
    .sort_values("total_sales", ascending=False)
)

print(product_sales)

  category  total_quantity  total_sales
3      스포츠             295     31743000
5     전자기기             259     26400000
2     생활용품             272     23915000
1       뷰티             223     23383000
4       식품             133     16573000
0       도서             149     16389000
6       패션             111     10587000
    product_id product_name category  total_quantity  total_sales
39          41   스포츠 상품 041      스포츠              35      5705000
11          12    식품 상품 012       식품              25      4375000
8            9   스포츠 상품 009      스포츠              20      3860000
70          72    뷰티 상품 072       뷰티              20      3780000
69          71  전자기기 상품 071     전자기기              23      3703000
..         ...          ...      ...             ...          ...
95          98   스포츠 상품 098      스포츠              13       130000
92          95  전자기기 상품 095     전자기기               6       120000
33          35   스포츠 상품 035      스포츠               2       118000
5            6  전자기기

# 10. 월별 고객별 요약표 만들기

In [118]:
monthly_sales = (
    completed_order_sales
    .groupby("order_month", as_index = False)
    .agg(
        total_sales = ("line_total", "sum"),
        order_count = ("order_id", "nunique"),
    )
    .sort_values("order_month")
)
print(monthly_sales)

   order_month  total_sales  order_count
0      2025-07      5869000            8
1      2025-08     15621000           18
2      2025-09     10190000           13
3      2025-10     25766000           26
4      2025-11      8812000           12
5      2025-12     11501000           14
6      2026-01     17423000           22
7      2026-02      9749000           17
8      2026-03     13429000           14
9      2026-04     17553000           23
10     2026-05      8063000           11
11     2026-06      2826000            4
12     2026-07      2188000            2


In [119]:
customer_sales = (
    completed_order_sales
    .groupby("customer_id", as_index=False)
    .agg(
        order_count=("order_id", "nunique"),
        total_sales=("line_total", "sum"),
    )
    .sort_values("total_sales", ascending=False)
)
print(customer_sales)

    customer_id  order_count  total_sales
76          117            5      4100000
62          102            4      3996000
51           83            4      3880000
21           30            5      3590000
29           40            4      3523000
..          ...          ...          ...
33           49            1       262000
10           15            1       252000
44           69            2       130000
50           80            1       118000
71          112            2       112000

[100 rows x 3 columns]


In [120]:
customer_sales = customer_sales.merge(
    customers[["customer_id", "city"]],
    on="customer_id",
    how="left",
    validate = "one_to_one",
)
print(customer_sales)

    customer_id  order_count  total_sales city
0           117            5      4100000   성남
1           102            4      3996000   고양
2            83            4      3880000   수원
3            30            5      3590000   서울
4            40            4      3523000   서울
..          ...          ...          ...  ...
95           49            1       262000   대구
96           15            1       252000   서울
97           69            2       130000   서울
98           80            1       118000   인천
99          112            2       112000   성남

[100 rows x 4 columns]


# 11. total_check

In [121]:
original_total = completed_sales_items["line_total"].sum()
category_total = category_sales["total_sales"].sum()
monthly_total = monthly_summary["total_sales"].sum()
customer_total = customer_sales["total_sales"].sum()

print("원본 completed line_total 합계:", original_total)
print("카테고리 합계:", category_total)
print("월별 합계:", monthly_total)
print("고객별 합계:", customer_total)

print(original_total == category_total)
print(original_total == monthly_total)
print(original_total == customer_total)

원본 completed line_total 합계: 148990000
카테고리 합계: 148990000
월별 합계: 148990000
고객별 합계: 148990000
True
True
True


# 12. 결과 CSV 저장하고 다시 읽기

In [122]:
print(get_project_root()/"report")

C:\dev\kant-axagent-study\llm-data-analysis-course\report


In [123]:
from pathlib import Path

REPORT_DIR = get_project_root() / "report"

# report 폴더가 없으면 생성
REPORT_DIR.mkdir(parents=True, exist_ok=True)

category_sales.to_csv(
    REPORT_DIR / "ch04_category_sales.csv",
    index=False
)

product_sales.to_csv(
    REPORT_DIR / "ch04_product_sales.csv",
    index=False
)

monthly_sales.to_csv(
    REPORT_DIR / "ch04_monthly_sales.csv",
    index=False
)

customer_sales.to_csv(
    REPORT_DIR / "ch04_customer_sales.csv",
    index=False
)

In [124]:
print("=== 카테고리별 ===")
print(category_sales)

print("\n=== 상품별 TOP 5 ===")
print(product_sales.head(5))

print("\n=== 월별 ===")
print(monthly_summary)

print("\n=== 고객별 TOP 5 ===")
print(customer_sales.head(5))

=== 카테고리별 ===
  category  total_quantity  total_sales
3      스포츠             295     31743000
5     전자기기             259     26400000
2     생활용품             272     23915000
1       뷰티             223     23383000
4       식품             133     16573000
0       도서             149     16389000
6       패션             111     10587000

=== 상품별 TOP 5 ===
    product_id product_name category  total_quantity  total_sales
39          41   스포츠 상품 041      스포츠              35      5705000
11          12    식품 상품 012       식품              25      4375000
8            9   스포츠 상품 009      스포츠              20      3860000
70          72    뷰티 상품 072       뷰티              20      3780000
69          71  전자기기 상품 071     전자기기              23      3703000

=== 월별 ===
   order_month  total_sales  order_count
0      2025-07      5869000            8
1      2025-08     15621000           18
2      2025-09     10190000           13
3      2025-10     25766000           26
4      2025-11      8812000       

# 13. 최종 Evidence 작성하기

[Chapter 04 Evidence]

1. Notebook
- notebooks/ch04_pandas_basic.ipynb 실행 완료: 예

2. 병합 검증
- orders.order_id 중복: 0
- orders merge 전 행 수: 764
- orders merge 후 행 수: 764
- orders merge 미매칭: 0
- products merge 전 행 수: 474
- products merge 후 행 수: 474
- products merge 미매칭: 0

3. 계산 범위
- 사용 주문 상태: completed
- line_total 계산식: quantity × unit_price
- 날짜 변환 실패: 0

4. 교차 검증
- 완료 주문 상세 line_total 합계: 148990000
- category_sales 합계: 148990000
- monthly_summary 합계: 148990000
- customer_sales 합계: 148990000
- 합계 일치 여부: PASS

5. 저장 파일
- ch04_category_sales.csv: 존재
- ch04_product_sales.csv: 존재
- ch04_monthly_sales.csv: 존재
- ch04_customer_sales.csv: 존재

6. LLM 검증
- LLM이 빠뜨린 검증 항목:
  - 이전 merge에서 생성된 `_merge` 컬럼이 이미 존재하는 상태를 고려하지 못했다.
  - 현재 DataFrame의 컬럼 상태를 먼저 확인하는 과정이 필요했다.
  - merge 코드가 문법적으로 맞더라도 실제 key 관계, 행 수 변화, 미매칭 여부를 직접 검증해야 했다.
  - 따라서 LLM이 제안한 코드는 수정 후 사용했다.

7. 남은 질문
- 스포츠 카테고리의 total_sales가 가장 높은 이유가 판매 수량이 많아서인지, 상품 단가가 높아서인지 추가로 확인하고 싶다.
- 2025-10의 total_sales와 order_count가 가장 높았던 이유가 특정 상품의 판매 증가 때문인지 전체 주문 수 증가 때문인지 확인하고 싶다.